# Analísis de Clustering - devalign-ml

Este cuaderno permite ejecutar el pipeline de clustering de habilidades técnicas usando **UMAP** y **HDBSCAN**, calcular el **Silhouette Score** y generar la visualización correspondiente. 

Puedes ejecutarlo localmente o cargarlo directamente en **Google Colab** desde tu repositorio de GitHub.

## 1. Configuración del Entorno (Solo para Google Colab)

Si estás corriendo este cuaderno en Google Colab, ejecuta la siguiente celda para clonar tu repositorio, instalar dependencias y añadir el directorio al PATH de ejecución de Python.

In [ ]:
# Verificar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    # Configura la URL de tu repositorio de GitHub aquí
    REPO_URL = "https://github.com/tu-usuario/devalign-ml.git"
    
    # Clonar el repositorio y moverse a la carpeta
    import os
    import sys
    repo_name = REPO_URL.split("/")[-1].replace(".git", "")
    if not os.path.exists(repo_name):
        !git clone {REPO_URL}
    
    os.chdir(repo_name)
    print(f"Directorio de trabajo: {os.getcwd()}")
    
    # Instalar dependencias requeridas
    !pip install -r requirements.txt
    
    # Agregar la carpeta al path de Python para importar los módulos de src
    sys.path.append(os.getcwd())
else:
    print("Corriendo en entorno local. Asegúrate de tener activado tu entorno virtual.")

## 2. Variables de Entorno y Credenciales

Carga tus variables de entorno para conectarte a la base de datos de Supabase y la API de Groq.

*Tip para Colab:* En lugar de subir un archivo `.env`, puedes ir a la barra lateral izquierda en Google Colab, dar clic en el ícono de llave (**Secrets** o **Claves**) e introducir `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY` y `GROQ_API_KEY` de forma segura.

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# Intentar cargar desde Secrets de Colab si aplica
try:
    from google.colab import userdata
    os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
    os.environ["SUPABASE_SERVICE_ROLE_KEY"] = userdata.get("SUPABASE_SERVICE_ROLE_KEY")
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Variables de entorno cargadas con éxito desde Colab Secrets.")
except:
    print("Buscando archivo .env local para credenciales...")

## 3. Carga y Procesamiento de Datos

Cargaremos los datos de habilidades y ofertas. La primera vez se descargarán de Supabase y se guardarán en la caché local (`data/raw_*.json`) para que las siguientes ejecuciones sean instantáneas sin saturar la base de datos.

In [ ]:
import numpy as np
import pandas as pd
import logging

# Configurar visualización de logs
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

from src.data_loader import load_raw_data, process_job_offer_vectors

# Cargar datos (intentando usar caché primero)
skills_data, offers_data = load_raw_data(use_cache=True)
valid_offers, alias_to_canonical = process_job_offer_vectors(skills_data, offers_data)

vectors = np.array([o["vector"] for o in valid_offers])
print(f"\nEmbeddings de ofertas listos para clustering: {vectors.shape}")

## 4. Ejecución del Clustering (UMAP + HDBSCAN)

Reduciremos las dimensiones e iniciaremos el algoritmo de clustering HDBSCAN. Veremos el Silhouette Score final.

In [ ]:
from src.clustering import ClusteringPipeline

pipeline = ClusteringPipeline()
results = pipeline.run(vectors)

print("\n" + "=" * 50)
print(f"METRICA - Silhouette Score (excl. ruido): {results['silhouette_score']:.4f}")
print("=" * 50)

## 5. Visualización Gráfica para el PPT

La gráfica de dispersión a continuación muestra la distribución bidimensional obtenida mediante UMAP y cómo HDBSCAN agrupa las ofertas de trabajo. Puedes tomarle captura de pantalla a este gráfico junto al Silhouette Score para tu presentación.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

X_reduced = results["X_reduced"]
labels = results["final_labels"]

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=X_reduced[:, 0], 
    y=X_reduced[:, 1], 
    hue=labels, 
    palette='viridis', 
    legend='full', 
    alpha=0.8,
    edgecolor='w',
    linewidth=0.5
)
plt.title(f"Clústeres UMAP + HDBSCAN (Silhouette Score: {results['silhouette_score']:.4f})")
plt.xlabel("UMAP Dimensión 1")
plt.ylabel("UMAP Dimensión 2")
plt.legend(title="Etiqueta Clúster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()